In [2]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep bincode = "1.3"

In [3]:
use std::fs;
use smartcore::tree::decision_tree_classifier::DecisionTreeClassifier;
use smartcore::linalg::basic::matrix::DenseMatrix;

In [9]:
let bytes = fs::read("model.bin").unwrap();
let dt_loaded: DecisionTreeClassifier<f64, i32, DenseMatrix<f64>, Vec<i32>> = bincode::deserialize(&bytes).unwrap();

In [5]:
:dep axum = { version = "0.7.3" }
:dep tokio = { version = "1.0", features = ["full"] }
:dep serde = { version = "1.0", features = ["derive"] }

In [6]:
use axum::{
    routing::get,
    Router,
    extract::{Query, Extension},
    Json,
};
use axum;
use std::sync::Arc;

In [10]:
use serde::Deserialize;
use axum::extract::State;
#[derive(Deserialize)]
struct RGB{
    r:f64,
    g:f64, 
    b:f64
}
async fn predict_handler(
    State(model): State<Arc<DecisionTreeClassifier<f64, i32, DenseMatrix<f64>, Vec<i32>>>>, // Заменили Extension на State
    Query(RGB { r, g, b }): Query<RGB>,
) -> Json<i32> {
    let input = DenseMatrix::from_2d_vec(&vec![vec![r, g, b]]);
    let res = model.predict(&input).unwrap();
    Json(res[0])
}


In [11]:
let shared_model = Arc::new(dt_loaded);

In [ ]:
use axum::Extension;
let app = Router::new()
    .route("/api/predict", get(predict_handler))
    .with_state(shared_model); 
let listener = tokio::net::TcpListener::bind("127.0.0.1:3000").await.unwrap();
axum::serve(listener, app).await.unwrap();
//http://127.0.0.1:3000/api/predict?r=141.0&g=250.0&b=0.0